In [ ]:
import requests
import pandas as pd
import trafilatura
from datetime import datetime, timedelta
from multiprocessing import Pool
import time
import feedparser

#API keys

GNEWS_KEY = "02228a1f03049d605978e65837944148"
NEWSAPI_KEY = "pub_b94859228b0a417e90d1a6fc38b0862d"

#Content extraction
def extract_article(url):
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        downloaded = trafilatura.fetch_url(url)
        text = trafilatura.extract(downloaded)

        if text and len(text) > 100:
            return text
    except:
        pass

    try:
        html = requests.get(url, headers=headers, timeout=10).text
        return html[:2000]
    except:
        return None

#Retry wrapper
def safe_extract(url):
    for _ in range(2):
        text = extract_article(url)
        if text:
            return text
    return None

#Gnews
def fetch_gnews():
    articles = []

    queries = [
    "Iran Israel war",
    "Iran Israel conflict",
    "Israel strikes Iran",
    "Iran retaliation Israel",
    "Middle East war",
    "Iran missile attack",
    "Israel airstrike Iran",
    "Iran nuclear tensions",
    "US Iran Israel war",
    "Gaza Iran Israel conflict",
    "Hezbollah Israel Iran",
    "Iran drone attack Israel",
    "Israel defence Iran war",
    "Iran oil crisis war",
    "Middle East escalation Iran Israel"
]

    for q in queries:
        for page in range(1, 6):
            try:
                url = f"https://gnews.io/api/v4/search?q={q}&lang=en&max=10&page={page}&apikey={GNEWS_KEY}"
                res = requests.get(url).json()

                if "articles" not in res:
                    continue

                for a in res["articles"]:
                    articles.append({
                        "headline": a.get("title"),
                        "url": a.get("url"),
                        "date": a.get("publishedAt"),
                        "source": a.get("source", {}).get("name")
                    })
            except:
                continue

            time.sleep(0.5)

    return articles

#Newsapi
def fetch_newsapi():
    articles = []

    queries = ["Iran Israel", "Middle East war"]
    today = datetime.today()

    for q in queries:
        for i in range(5):
            date = (today - timedelta(days=i)).strftime("%Y-%m-%d")

            for page in range(1, 6):
                try:
                    url = f"https://newsapi.org/v2/everything?q={q}&from={date}&to={date}&language=en&pageSize=20&page={page}&apiKey={NEWSAPI_KEY}"
                    res = requests.get(url).json()

                    if "articles" not in res:
                        continue

                    for a in res["articles"]:
                        articles.append({
                            "headline": a.get("title"),
                            "url": a.get("url"),
                            "date": a.get("publishedAt"),
                            "source": a.get("source", {}).get("name")
                        })
                except:
                    continue

                time.sleep(0.5)

    return articles

#Google news rss
import urllib.parse



def fetch_google_rss():
    articles = []

    queries = ["Iran Israel war", "Middle East conflict"]

    date_ranges = [
        "after:2026-03-01 before:2026-03-05",
        "after:2026-03-05 before:2026-03-10",
        "after:2026-03-10 before:2026-03-15",
        "after:2026-03-15 before:2026-03-20",
    ]

    for q in queries:
        for d in date_ranges:
            full_query = f"{q} {d}"
            encoded_q = urllib.parse.quote(full_query)

            url = f"https://news.google.com/rss/search?q={encoded_q}&hl=en&gl=US&ceid=US:en"
            feed = feedparser.parse(url)

            for entry in feed.entries:
                articles.append({
                    "headline": entry.title,
                    "url": entry.link,
                    "date": entry.get("published", None),
                    "source": "Google News"
                })

    return articles
#Fetch all data
articles = []
articles += fetch_gnews()
articles += fetch_newsapi()
articles += fetch_google_rss()   # 🔥 NEW

#Dataframe
df = pd.DataFrame(articles)

df = df.drop_duplicates(subset=["url"])
df = df[df["url"].notnull()]
df = df.reset_index(drop=True)

print("Collected BEFORE content:", len(df))

#Remove blocked domains
blocked = ["bloomberg.com", "timesofisrael.com"]
df = df[~df["url"].str.contains("|".join(blocked))]


#Parallel extraction
with Pool(5) as p:
    df["content"] = p.map(safe_extract, df["url"])

#Clean
df = df[df["content"].notnull()]
df = df[df["content"].str.len() > 100]

#Final
df["last_updated"] = datetime.now()

print("\nFINAL DATASET SIZE:", len(df))
print(df.head())

df.to_csv("iran_israel_news_dataset.csv", index=False)

Collected BEFORE content: 897


ERROR:trafilatura.downloads:not a 200 response: 401 for URL https://www.reuters.com/business/unilever-imposes-global-hiring-freeze-citing-middle-east-war-effects-memo-says-2026-03-30/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://seekingalpha.com/article/4887174-the-ceasefire-is-slipping-away
ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.lbc.co.uk/article/c30ccfe0946a4a09bb57fb13552d2b3b-5HjdX5g_2/
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.telegraphindia.com/world/us-israel-vs-iran-is-not-world-war-iii-but-the-past-could-hold-keys-to-the-conflicts-future/cid/2151313
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.ndtvprofit.com/world/golden-pass-lng-project-qatarenergy-s-largest-energy-investment-in-the-united-states-achieves-historic-milestone-with-first-lng-output-11287603
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.ndtv.com/world-news/hezbollah-israel-war


FINAL DATASET SIZE: 889
                                            headline  \
0  Trump says US negotiating with Iran's parliame...   
1  U.S.-Israel-Iran war puts India’s EV gap in focus   
2  Democratic lawmaker says US ‘badly and embarra...   
3  Iran-Israel war LIVE updates: Kuwait says Iran...   
4  Trump again threatens widespread destruction i...   

                                                 url                  date  \
0  https://economictimes.indiatimes.com/news/inte...  2026-03-31T01:31:00Z   
1  https://www.thehindu.com/data/us-israel-iran-w...  2026-03-31T01:30:00Z   
2  https://www.moneycontrol.com/world/democratic-...  2026-03-31T01:26:05Z   
3  https://www.thehindu.com/news/international/us...  2026-03-31T01:13:44Z   
4  https://www.ajc.com/news/2026/03/trump-again-t...  2026-03-31T01:01:13Z   

                             source  \
0                The Economic Times   
1                         The Hindu   
2                      Moneycontrol   
3            